# 🛡️ GRACE H2: Đánh Giá Tập Validation Trên Vertex AI (DeepSeek-V4.1-Flash)

**Môi Trường Thực Thi**: Kaggle Cloud (CPU / Standard VM) với Google Cloud Vertex AI Model-as-a-Service (MaaS)  
**Hợp Đồng Đánh Giá**: `evaluation_contract.yaml`  
**Mô Hình Nền (Backbone LLM)**: `deepseek-ai/deepseek-v4.1-flash-maas` (Location: `global`)  
**Bộ Dữ Liệu**: Validation Split của Devign & ReVeal  

### Mục Tiêu Thực Nghiệm & Nguyên Tắc Vận Hành:
1. **Nạp 100% Dữ Liệu Thật**: Tự động tìm kiếm và nạp đầy đủ 100% các prompt thật từ tập validation (`devign` và `reveal`), tuyệt đối không dùng mock/synthetic prompts.
2. **Không Xung Đột Gói Thư Viện**: Tuyệt đối không chạy `pip install -q --upgrade ...` để bảo toàn môi trường phụ thuộc ổn định của Kaggle.
3. **Parser Chuẩn Canonical**: Sử dụng `ExactLastLineBinaryParser` chuẩn hóa từ `strict_parser.py`, không dùng regex phức tạp gây lỗi cú pháp.
4. **Xác Thực Khép Kín**: Nạp Service Account Key từ Kaggle User Secrets (`GCP_SA_KEY`), tự động bắt HTTP 401 để refresh access token.
5. **Đối Chiếu Hiệu Năng Với Baselines**: Tính toán đầy đủ Accuracy, Precision, Recall, F1, MCC, Balanced Accuracy đối chiếu với B2 (1-shot) và B3 (2-shot Contrastive).
6. **Chuẩn Hóa Kết Quả**: Xuất `predictions.jsonl`, `raw_responses.jsonl`, `run_manifest.json`, `evaluation.json`, và nén thành `H2_vertex_validation_kaggle_results.zip`.

## Bước 1: Nạp GCP Service Account Key & Cấu Hình Môi Trường
Đọc Service Account Key từ Kaggle Secret `GCP_SA_KEY`, ghi ra file `/tmp/gcp_sa_key.json`,
thiết lập `GOOGLE_APPLICATION_CREDENTIALS` và kiểm tra sinh Access Token hợp lệ qua `google.auth`.

In [ ]:
import os
import sys
import json
import time
import shutil
from pathlib import Path

PROJECT_ID = "grace-research-2903"
VERTEX_LOCATION = "global"
MODEL_ID = "deepseek-ai/deepseek-v4.1-flash-maas"
CONTEXT_LIMIT_BYTES = 8192

# 1. Nap Secret tu Kaggle User Secrets
sa_path = Path("/tmp/gcp_sa_key.json")

try:
    from kaggle_secrets import UserSecretsClient
    print("[*] Dang nap secret 'GCP_SA_KEY' tu Kaggle User Secrets...")
    user_secrets = UserSecretsClient()
    sa_json_str = user_secrets.get_secret("GCP_SA_KEY")
    with open(sa_path, "w", encoding="utf-8") as f:
        f.write(sa_json_str.strip())
    os.chmod(sa_path, 0o600)
    print("✓ Da ghi Service Account key thanh cong tai /tmp/gcp_sa_key.json")
except Exception as e:
    print(f"[!] Khong nap duoc tu Kaggle Secrets ({e}), kiem tra fallback keys...")
    fallback_keys = [Path(".gcp_key.json"), Path("/kaggle/working/.gcp_key.json"), Path("../.gcp_key.json")]
    for fk in fallback_keys:
        if fk.exists():
            shutil.copy2(fk, sa_path)
            os.chmod(sa_path, 0o600)
            print(f"✓ Su dung fallback key tu: {fk}")
            break

assert sa_path.exists(), "FATAL: Khong tim thay Service Account Key! Vui long cau hinh secret GCP_SA_KEY trong Kaggle Notebook Settings (Add-ons -> Secrets)."

# 2. Thiet lap bien moi truong GCP
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(sa_path)
os.environ["CLOUD_ML_PROJECT_ID"] = PROJECT_ID
os.environ["GCP_PROJECT"] = PROJECT_ID
os.environ["VERTEX_LOCATION"] = VERTEX_LOCATION

# 3. Xac thuc sinh Access Token
import google.auth
import google.auth.transport.requests

scopes = ["https://www.googleapis.com/auth/cloud-platform"]
credentials, auth_project = google.auth.default(scopes=scopes)
auth_req = google.auth.transport.requests.Request()
credentials.refresh(auth_req)

assert credentials.token, "FATAL: Khong the sinh OAuth2 Access Token tu Service Account!"
print(f"✓ [Xac Thuc Thanh Cong] Token hop le: {credentials.token[:12]}... (Project: {auth_project or PROJECT_ID})")
print(f"✓ Vertex Location: {VERTEX_LOCATION} | Model: {MODEL_ID}")

## Bước 2: Chuẩn Bị Workspace & Cấu Trúc Thư Mục Kết Quả
Kaggle đã tích hợp sẵn `requests`, `google-auth`, `pandas`, `scikit-learn`, `numpy`, `tqdm`.
Tuyệt đối không chạy `pip install -q --upgrade` để tránh xung đột resolver.

In [ ]:
import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score,
    f1_score, matthews_corrcoef, balanced_accuracy_score, accuracy_score
)

WORKDIR = Path("/kaggle/working/h2_vertex_validation")
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Thu muc lam viec: {WORKDIR.resolve()}")

## Bước 3: Động Cơ Giao Tiếp Vertex AI & Strict Canonical Parser
Tuân thủ nghiêm ngặt theo hợp đồng `evaluation_contract.yaml`:
- Parameters: `temperature=0.0`, `seed=42`, `max_tokens=256`, `stream=False`
- Truncation Guard: 8,192 UTF-8 bytes
- Strict Parser: `ExactLastLineBinaryParser` theo mã nguồn chuẩn tắc của `strict_parser.py`
- Cơ chế tự động bắt HTTP 401 để refresh access token và exponential backoff khi gặp lỗi.

In [ ]:
import hashlib

class ExactLastLineBinaryParser:
    """Strict parser conforming to evaluation_contract.yaml exact_last_line_binary."""
    @staticmethod
    def parse(raw_text: str):
        if not isinstance(raw_text, str) or not raw_text.strip():
            return None, "invalid"
        
        last_line = next(
            (line.strip() for line in reversed(raw_text.splitlines()) if line.strip()),
            "",
        )
        if last_line == "1":
            return 1, "valid"
        if last_line == "0":
            return 0, "valid"
        return None, "invalid"

class VertexAIDeepSeekClient:
    """Vertex AI MaaS Client for DeepSeek-V4.1-Flash."""
    def __init__(self, project_id: str, location: str = "global", model_id: str = MODEL_ID):
        self.project_id = project_id
        self.location = location
        self.model_id = model_id
        self.url = f"https://aiplatform.googleapis.com/v1/projects/{self.project_id}/locations/{self.location}/endpoints/openapi/chat/completions"
        self.scopes = ["https://www.googleapis.com/auth/cloud-platform"]
        self.credentials, _ = google.auth.default(scopes=self.scopes)
        self.auth_req = google.auth.transport.requests.Request()
        self.session = requests.Session()
        
    def get_valid_token(self, force_refresh: bool = False):
        if force_refresh or not self.credentials.valid:
            self.credentials.refresh(self.auth_req)
        return self.credentials.token
    
    def request(self, prompt_text: str, max_retries: int = 5):
        prompt_bytes = prompt_text.encode('utf-8')
        if len(prompt_bytes) > CONTEXT_LIMIT_BYTES:
            prompt_text = prompt_bytes[:CONTEXT_LIMIT_BYTES].decode('utf-8', errors='ignore')

        payload = {
            "model": self.model_id,
            "temperature": 0.0,
            "seed": 42,
            "max_tokens": 256,
            "stream": False,
            "messages": [{"role": "user", "content": prompt_text}]
        }
        
        attempts = []
        start_overall = time.perf_counter()

        for attempt in range(1, max_retries + 1):
            start_attempt = time.perf_counter()
            token = self.get_valid_token(force_refresh=(attempt > 1))
            headers = {
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json"
            }

            try:
                resp = self.session.post(self.url, headers=headers, json=payload, timeout=60)
                elapsed_ms = (time.perf_counter() - start_attempt) * 1000
                
                attempt_record = {
                    "attempt": attempt,
                    "started_unix": time.time() - (elapsed_ms / 1000),
                    "http_status": resp.status_code,
                    "elapsed_ms": round(elapsed_ms, 2),
                    "response_text": resp.text
                }
                attempts.append(attempt_record)

                if resp.status_code == 200:
                    resp_json = resp.json()
                    content = resp_json.get("choices", [{}])[0].get("message", {}).get("content", "")
                    parsed_val, status = ExactLastLineBinaryParser.parse(content)
                    
                    return {
                        "status": status,
                        "prediction": parsed_val,
                        "raw_response": content,
                        "elapsed_ms": round((time.perf_counter() - start_overall) * 1000, 2),
                        "attempts": attempts,
                        "usage": resp_json.get("usage", {}),
                        "request_body": payload
                    }
                elif resp.status_code == 401:
                    print(f"[*] HTTP 401 Unauthorized, refreshing OAuth2 token for attempt {attempt}...")
                    self.credentials.refresh(self.auth_req)
                    time.sleep(1)
                else:
                    backoff_sec = min(2 ** attempt, 30)
                    time.sleep(backoff_sec)
                    
            except Exception as e:
                elapsed_ms = (time.perf_counter() - start_attempt) * 1000
                attempts.append({
                    "attempt": attempt,
                    "error": str(e),
                    "elapsed_ms": round(elapsed_ms, 2)
                })
                time.sleep(min(2 ** attempt, 30))

        return {
            "status": "exhausted",
            "prediction": None,
            "raw_response": "",
            "elapsed_ms": round((time.perf_counter() - start_overall) * 1000, 2),
            "attempts": attempts,
            "request_body": payload
        }

client = VertexAIDeepSeekClient(PROJECT_ID, VERTEX_LOCATION, MODEL_ID)
print("✓ Client VertexAIDeepSeekClient da khoi tao san sang!")

## Bước 4: Smoke Test Kết Nối Vertex AI DeepSeek
Gửi test query thực tế để kiểm tra khả năng suy luận, đo lường latency và xác thực định dạng nhị phân.

In [ ]:
smoke_prompt = """### TASK INSTRUCTION
You are an expert C/C++ software security researcher.
Determine whether the target function contains a software vulnerability.

### TARGET CODE
int check_buffer(char *buf, int size) {
    if (size <= 0) return -1;
    char dest[64];
    if (size < 64) {
        memcpy(dest, buf, size);
        return 0;
    }
    return -1;
}

### OUTPUT CONTRACT
The last non-empty line of your response must contain exactly one character: `1` if the target is Vulnerable, or `0` if the target is Non-vulnerable. Do not output any other label or score.
"""

print("[*] Dang thuc hien Smoke Test tren Vertex AI...")
smoke_res = client.request(smoke_prompt)

print(f"✓ HTTP Status: {smoke_res.get('status')}")
print(f"✓ Raw Output: {smoke_res.get('raw_response')}")
print(f"✓ Parsed Prediction: {smoke_res.get('prediction')} (Status: {smoke_res.get('status')})")
print(f"⚡ Latency: {smoke_res.get('elapsed_ms')} ms")

assert smoke_res.get("status") == "valid", f"Smoke test khong hop le: {smoke_res}"
assert smoke_res.get("prediction") in [0, 1], f"Nhan khong hop le: {smoke_res.get('prediction')}"
print("🎉 [SMOKE TEST PASSED] Ket noi Vertex AI hoan hao!")

## Bước 5: Nạp 100% Dữ Liệu Prompts Tập Validation
Hệ thống tự động giải nén các tệp đính kèm và tìm kiếm tệp `prompts.jsonl` hoặc `prompts.jsonl.gz`
của tập validation (`devign` và `reveal`).
**Cam kết kỹ thuật**: Bắt buộc nạp 100% dữ liệu thực tế, tuyệt đối không dùng mock/synthetic prompts rút gọn.

In [ ]:
import gzip
import zipfile

def unpack_archives_if_needed():
    """Tu dong giai nen cac file zip chua dataset neu co."""
    for p in Path("/kaggle/input").rglob("*.zip"):
        extract_dir = Path("/kaggle/working/unpacked") / p.stem
        if not extract_dir.exists():
            print(f"[*] Dang giai nen archive: {p} -> {extract_dir}")
            extract_dir.mkdir(parents=True, exist_ok=True)
            try:
                with zipfile.ZipFile(p, "r") as zf:
                    zf.extractall(extract_dir)
                print(f"✓ Da giai nen {p.name}")
            except Exception as e:
                print(f"[!] Loi khi giai nen {p.name}: {e}")

unpack_archives_if_needed()

def load_validation_prompts(dataset_name: str, limit: int = None):
    """
    Nap 100% du lieu prompt that cua tap validation tu cac nguon dataset Kaggle hoac unpacked archive.
    Bat buoc 100% du lieu thuc te.
    """
    candidates = [
        Path(f"/kaggle/input/grace-h2-vertex-validation/inputs/{dataset_name}/prompts.jsonl"),
        Path(f"/kaggle/input/grace-h2-vertex-validation/inputs/{dataset_name}/prompts.jsonl.gz"),
        Path(f"/kaggle/input/grace-h2-vertex-validation/{dataset_name}/prompts.jsonl"),
        Path(f"/kaggle/input/grace-h2-vertex-validation/{dataset_name}/prompts.jsonl.gz"),
        Path(f"/kaggle/input/h2-vertex-validation/inputs/{dataset_name}/prompts.jsonl"),
        Path(f"/kaggle/input/h2-vertex-validation/inputs/{dataset_name}/prompts.jsonl.gz"),
        Path(f"/kaggle/input/validation/inputs/{dataset_name}/prompts.jsonl"),
        Path(f"/kaggle/input/validation/inputs/{dataset_name}/prompts.jsonl.gz"),
        Path(f"/kaggle/working/unpacked/inputs/{dataset_name}/prompts.jsonl"),
        Path(f"/kaggle/working/unpacked/{dataset_name}/prompts.jsonl"),
        Path(f"/kaggle/working/GRACE/inputs/{dataset_name}/prompts.jsonl"),
        Path(f"/kaggle/working/GRACE/inputs/{dataset_name}/prompts.jsonl.gz"),
        Path(f"./inputs/{dataset_name}/prompts.jsonl"),
        Path(f"./inputs/{dataset_name}/prompts.jsonl.gz")
    ]
    
    found_path = None
    for c in candidates:
        if c.exists():
            found_path = c
            break
            
    if not found_path:
        search_roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
        for root_dir in search_roots:
            if not root_dir.exists():
                continue
            for p in root_dir.rglob("*.jsonl*"):
                p_str = str(p).lower().replace("\\", "/")
                if dataset_name.lower() in p_str and "prompt" in p.name.lower():
                    found_path = p
                    break
            if found_path:
                break
            
    if not found_path:
        raise FileNotFoundError(
            f"FATAL: Khong tim thay file prompt that cho tap validation cua '{dataset_name}'!\n"
            f"Vui long dam bao Kaggle Dataset 'huuhieu3333/grace-h2-vertex-validation' da duoc gan vao Notebook,\n"
            f"hoac file zip du lieu validation da duoc upload vao /kaggle/input.\n"
            f"Tuan thu quy tac nghiem ngat: KHONG dung mock prompt rut gon!"
        )

    print(f"✓ [TIM THAY DATASET] Nguon: {found_path}")
    records = []
    open_fn = gzip.open if str(found_path).endswith(".gz") else open
    with open_fn(found_path, "rt", encoding="utf-8") as f:
        for line in f:
            line_str = line.strip()
            if not line_str:
                continue
            rec = json.loads(line_str)
            if "prompt_text" not in rec:
                for k in ["prompt", "input", "text", "content"]:
                    if k in rec and rec[k]:
                        rec["prompt_text"] = rec[k]
                        break
            if "ground_truth" not in rec or rec["ground_truth"] is None:
                for k in ["target", "label", "target_id", "vul"]:
                    if k in rec and rec[k] is not None:
                        try:
                            rec["ground_truth"] = int(rec[k])
                            break
                        except (ValueError, TypeError):
                            pass
            records.append(rec)
            
    if not records:
        raise ValueError(f"FATAL: Tep tin prompt '{found_path}' rong hoac khong chua ban ghi hop le!")

    total_found = len(records)
    if limit is not None and limit > 0:
        records = records[:limit]
        print(f"✓ Da nap {len(records)}/{total_found} mau cho tap validation {dataset_name.upper()} (ap dung limit={limit})")
    else:
        print(f"✓ Da nap TOAN BO {len(records)} mau that cho tap validation {dataset_name.upper()} (100% du lieu)")

    return records

# Mac dinh: EVAL_LIMIT = None (chay 100% toan bo cac mau cua tap validation)
EVAL_LIMIT = None
print("[*] San sang thuc hien danh gia validation cho Devign & ReVeal...")

## Bước 6: Thực Thi Đánh Giá (Evaluation Loop) & Xuất Chuẩn Hóa
Chạy toàn bộ các mẫu validation, ghi nhận thời gian thực thi (elapsed ms),
xuất `predictions.jsonl`, `raw_responses.jsonl`, và `run_manifest.json` cho từng dataset.

In [ ]:
def evaluate_dataset(dataset_name: str, records: list):
    """Thuc thi danh gia va xuat toan bo artifact chuan cho dataset."""
    out_dir = WORKDIR / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)
    
    pred_file = out_dir / "predictions.jsonl"
    raw_file = out_dir / "raw_responses.jsonl"
    
    preds_out = []
    raws_out = []
    
    print(f"\n=======================================================")
    print(f"🚀 Bat dau danh gia {len(records)} mau cho {dataset_name.upper()}")
    print(f"=======================================================")
    
    start_eval_time = time.perf_counter()
    
    with open(pred_file, "w", encoding="utf-8") as f_pred, open(raw_file, "w", encoding="utf-8") as f_raw:
        for rec in tqdm(records, desc=f"Evaluating {dataset_name}"):
            rec_id = rec.get("record_id", rec.get("query_id", "unknown"))
            prompt_content = rec.get("prompt_text", rec.get("prompt", ""))
            p_sha = hashlib.sha256(prompt_content.encode('utf-8')).hexdigest()
            
            res = client.request(prompt_content)
            
            pred_record = {
                "record_id": rec_id,
                "query_id": rec.get("query_id", rec_id),
                "code_sha256": rec.get("code_sha256", ""),
                "prompt_sha256": p_sha,
                "prediction": res.get("prediction"),
                "request_status": res.get("status"),
                "ground_truth": rec.get("ground_truth"),
                "arm": "H2-guarded-v3-validation",
                "elapsed_ms": res.get("elapsed_ms")
            }
            
            raw_record = {
                "record_id": rec_id,
                "query_id": rec.get("query_id", rec_id),
                "prompt_sha256": p_sha,
                "result": res
            }
            
            f_pred.write(json.dumps(pred_record) + "\n")
            f_raw.write(json.dumps(raw_record) + "\n")
            f_pred.flush()
            f_raw.flush()
            
            preds_out.append(pred_record)
            raws_out.append(raw_record)
            
    total_sec = time.perf_counter() - start_eval_time
    
    manifest = {
        "provider": "Vertex AI",
        "url": client.url,
        "parameters": {
            "model": MODEL_ID,
            "temperature": 0.0,
            "seed": 42,
            "max_tokens": 256,
            "stream": False
        },
        "context_limit_utf8_bytes": CONTEXT_LIMIT_BYTES,
        "parser": "exact_last_line_binary_canonical",
        "dataset": dataset_name,
        "split": "validation",
        "total_samples": len(records),
        "total_elapsed_seconds": round(total_sec, 2),
        "environment": "Kaggle Cloud (Vertex AI MaaS)"
    }
    with open(out_dir / "run_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
        
    print(f"✓ Hoan tat danh gia {dataset_name.upper()} trong {total_sec:.2f}s ({len(preds_out)} requests)!")
    return preds_out, raws_out

# 1. Danh gia Devign Validation
devign_preds, devign_raws = [], []
try:
    devign_records = load_validation_prompts("devign", limit=EVAL_LIMIT)
    devign_preds, devign_raws = evaluate_dataset("devign", devign_records)
except Exception as e:
    print(f"[!] Loi/Thong bao khi danh gia Devign: {e}")

# 2. Danh gia ReVeal Validation
reveal_preds, reveal_raws = [], []
try:
    reveal_records = load_validation_prompts("reveal", limit=EVAL_LIMIT)
    reveal_preds, reveal_raws = evaluate_dataset("reveal", reveal_records)
except Exception as e:
    print(f"[!] Loi/Thong bao khi danh gia ReVeal: {e}")

# Guard kiem tra nghiem ngat: bat buoc it nhat 1 tap du lieu validation phai duoc danh gia thanh cong
assert len(devign_preds) > 0 or len(reveal_preds) > 0, (
    "FATAL: Khong co mau prompt validation nao duoc danh gia thanh cong! "
    "Vui long kiem tra lai dataset attachment hoac file prompt trong /kaggle/input."
)

## Bước 7: Tính Toán Metrics & Đối Chiếu Baselines (B2 & B3)
Tính toán Accuracy, Precision, Recall, F1, MCC, Balanced Accuracy theo chuẩn `evaluation_contract.yaml`
và lập bảng so sánh đối chiếu trực tiếp với B2 (1-shot) và B3 (2-shot Contrastive).

In [ ]:
baselines = {
    "B2_baseline_1shot": {
        "f1": 0.3614,
        "recall": 0.2778,
        "precision": 0.5172,
        "accuracy": 0.4700,
        "mcc": -0.0292,
        "balanced_accuracy": 0.4867
    },
    "B3_contrastive_2shot": {
        "f1": 0.5500,
        "recall": 0.6111,
        "precision": 0.5000,
        "accuracy": 0.4600,
        "mcc": -0.1118,
        "balanced_accuracy": 0.4469
    }
}

def compute_and_report_metrics(dataset_name: str, preds: list):
    """Tinh toan metrics theo chuan evaluation_contract.yaml."""
    out_dir = WORKDIR / dataset_name
    valid_samples = [p for p in preds if p.get("request_status") == "valid" and p.get("ground_truth") is not None]
    
    if valid_samples:
        y_true = [int(s["ground_truth"]) for s in valid_samples]
        y_pred = [int(s["prediction"]) for s in valid_samples]
        
        acc = float(accuracy_score(y_true, y_pred))
        prec = float(precision_score(y_true, y_pred, zero_division=0))
        rec = float(recall_score(y_true, y_pred, zero_division=0))
        f1 = float(f1_score(y_true, y_pred, zero_division=0))
        mcc = float(matthews_corrcoef(y_true, y_pred)) if len(set(y_true)) > 1 and len(set(y_pred)) > 1 else 0.0
        bacc = float(balanced_accuracy_score(y_true, y_pred)) if len(set(y_true)) > 1 else acc
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    else:
        acc, prec, rec, f1, mcc, bacc = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
        tn, fp, fn, tp = 0, 0, 0, 0

    print("\n" + "=" * 88)
    print(f"📊 KET QUA DANH GIA VALIDATION: {dataset_name.upper()} ({len(valid_samples)}/{len(preds)} samples)")
    print("=" * 88)
    print(f"{'Metric':<22} | {'B2 (1-shot)':<12} | {'B3 (Contrastive)':<16} | {'H2 Validation':<20}")
    print("-" * 88)
    print(f"{'Accuracy':<22} | {baselines['B2_baseline_1shot']['accuracy']:<12.4f} | {baselines['B3_contrastive_2shot']['accuracy']:<16.4f} | {acc:<20.4f}")
    print(f"{'Precision':<22} | {baselines['B2_baseline_1shot']['precision']:<12.4f} | {baselines['B3_contrastive_2shot']['precision']:<16.4f} | {prec:<20.4f}")
    print(f"{'Recall':<22} | {baselines['B2_baseline_1shot']['recall']:<12.4f} | {baselines['B3_contrastive_2shot']['recall']:<16.4f} | {rec:<20.4f}")
    print(f"{'F1-Score':<22} | {baselines['B2_baseline_1shot']['f1']:<12.4f} | {baselines['B3_contrastive_2shot']['f1']:<16.4f} | {f1:<20.4f}")
    print(f"{'MCC':<22} | {baselines['B2_baseline_1shot']['mcc']:<12.4f} | {baselines['B3_contrastive_2shot']['mcc']:<16.4f} | {mcc:<20.4f}")
    print(f"{'Balanced Accuracy':<22} | {baselines['B2_baseline_1shot']['balanced_accuracy']:<12.4f} | {baselines['B3_contrastive_2shot']['balanced_accuracy']:<16.4f} | {bacc:<20.4f}")
    print("-" * 88)
    print(f"Confusion Matrix: TP={tp}, TN={tn}, FP={fp}, FN={fn}")
    print("=" * 88)

    eval_doc = {
        "metrics": {
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "mcc": mcc,
            "balanced_accuracy": bacc,
            "confusion_matrix": {"tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)},
            "n_valid": len(valid_samples),
            "n_total": len(preds)
        }
    }
    with open(out_dir / "evaluation.json", "w", encoding="utf-8") as f:
        json.dump(eval_doc, f, indent=2)
        
    return eval_doc

devign_eval = compute_and_report_metrics("devign", devign_preds) if devign_preds else {}
reveal_eval = compute_and_report_metrics("reveal", reveal_preds) if reveal_preds else {}

# Luu tong hop so sanh baseline
comp_report = {
    "baselines": baselines,
    "validation_results": {
        "devign": devign_eval,
        "reveal": reveal_eval
    }
}
with open(WORKDIR / "baseline_comparison_report.json", "w", encoding="utf-8") as f:
    json.dump(comp_report, f, indent=2, ensure_ascii=False)
print("✓ Bao cao tong hop da luu tai baseline_comparison_report.json")

## Bước 8: Đóng Gói Toàn Bộ Kết Quả Ra File ZIP
Nén toàn bộ thư mục `h2_vertex_validation` thành file `H2_vertex_validation_kaggle_results.zip`.

In [ ]:
zip_dest = Path("/kaggle/working/H2_vertex_validation_kaggle_results.zip")
print(f"[*] Dang dong goi toan bo ket qua vao: {zip_dest}...")

with zipfile.ZipFile(zip_dest, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(WORKDIR):
        for f in files:
            p = Path(root) / f
            zf.write(p, arcname=str(p.relative_to(WORKDIR)))
            
zip_mb = zip_dest.stat().st_size / (1024 * 1024)
print(f"🎉 [HOAN TAT] File ket qua: {zip_dest} ({zip_mb:.2f} MB)")